CHATBOT EVALUATION

In [25]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

creating the datapoints
- for a particular input -> what will be the output

In [26]:
from langsmith import Client
client = Client()

# defining dataset --> test data
dataset_name = "Chatbots Evaluation"
dataset = client.create_dataset(dataset_name)

client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain ?"},
            "outputs": {"answer": "A framework for building LLM applications"}
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['3b14bf57-fa7c-4145-ab36-e6c317cfc64b',
  'e69ed339-a9d6-486c-86ae-b97404458fe8',
  '4ef3550b-7b93-4d8c-9e77-4a3b8ff086b2',
  '2011fd9a-33cc-4e80-b556-78587fe6f066',
  'ab3b1076-d969-4560-b819-6792c0c5c8bb'],
 'count': 5,
 'as_of': '2026-08-01T12:46:28.020433659Z'}

metrics for eval - llm as judge

- correctness - metric

In [32]:
from openai import OpenAI
from langsmith import wrappers
groq_client = wrappers.wrap_openai(
    OpenAI(
        api_key=GROQ_API_KEY,
        base_url="https://api.groq.com/openai/v1"
    )
)
instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs:dict, outputs:dict, reference_outputs:dict) -> bool:
    user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0,
        messages = [
            {"role":"system","content":instructions},
            {"role":"user","content":user_content}
        ]
    ).choices[0].message.content

    return response == "CORRECT"

- concision - checks whether actual output is less than 2x the length of expected result

In [33]:
def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

Running Evaluations

In [34]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
# main chatbot function -> needs to be called for every input
def my_app(question: str,model: str="llama-3.3-70b-versatile",instructions: str = default_instructions) -> str:
    return groq_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role":"system","content":instructions},
            {"role":"user","content":question}
        ]
    ).choices[0].message.content

In [43]:
from typing import Dict, Any
def ls_target(inputs: Dict[str, Any]) -> Dict[str, str]:
    return {
        "response": my_app(inputs["question"])
    }

In [37]:
experiment_results = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="llama-3.3-70b-versatile"
)

View the evaluation results for experiment: 'llama-3.3-70b-versatile-0326b9a9' at:
https://smith.langchain.com/o/4a7e1065-23d2-4ab2-8bb4-13a5400bfe0e/datasets/7ee2c440-12da-4f4f-a8b7-a79dafa34c50/compare?selectedSessions=69570287-a072-4905-811c-fc8cb2cd3861




0it [00:00, ?it/s]

another experiment using diff model

In [44]:
from typing import Dict, Any
def ls_target1(inputs: Dict[str, Any]) -> Dict[str, str]:
    return {
        "response": my_app(inputs["question"],model="llama-3.1-8b-instant")
    }

In [45]:
experiment_results = client.evaluate(
    ls_target1,
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="llama-3.1-8b-instant"
)

View the evaluation results for experiment: 'llama-3.1-8b-instant-7c465f07' at:
https://smith.langchain.com/o/4a7e1065-23d2-4ab2-8bb4-13a5400bfe0e/datasets/7ee2c440-12da-4f4f-a8b7-a79dafa34c50/compare?selectedSessions=434fe53e-eb81-4d58-b33c-aaab15781ff0




0it [00:00, ?it/s]